#Load_Model

*Propósito:* Cargar el modelo entrenado (pickle) desde el repositorio y registrarlo en Unity Catalog.

In [0]:
%run ../config/config

In [0]:
%%capture
%pip install catboost optuna xgboost lightgbm openpyx1 category_encoders fsspec

In [0]:
import sys
import time
import pickle
import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.catboost
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature

sys.path.insert(0, str(PROJECT_PATH))
from utils.logger import MLOpsLogger

In [0]:
%skip
import pickle
import os

project_path = "/Workspace/Users/marioavento@bcp.com.pe/cemm-202605-dtr-deploy-troncal-bhv-pyme/src/data"
model_path = os.path.join(project_path, "models", "model_v2_REDUCC_32V.pkl")

with open(model_path, "rb") as f:
    model = pickle.load(f)

print("=== TIPO DE MODELO ===")
print(type(model))
print("\n=== MÓDULO ===")
print(model.__class__.__module__)
print("\n=== ¿TIENE predict_proba? ===")
print(hasattr(model, 'predict_proba'))

# Si el modelo expone parámetros (CatBoost, XGBoost, LightGBM, sklearn)
if hasattr(model, 'get_params'):
    print("\n=== PARÁMETROS (primeros 5) ===")
    params = model.get_params()
    for i, (k, v) in enumerate(params.items()):
        if i >= 5:
            break
        print(f"{k}: {v}")

In [0]:
logger = MLOpsLogger(
    name='01_load_model',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': '01_load_model'
    }
)

In [0]:
mlflow.set_registry_uri("databricks-uc")
logger.info(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
model_name = REGISTERED_MODEL_NAME

print(f"Buscando modelo: {model_name}\n")

try:
    versions = client.search_model_versions(f"name='{model_name}'")
    if not versions:
        print("❌ El modelo NO está registrado en Unity Catalog.")
    else:
        print(f"✅ Modelo encontrado con {len(versions)} versión(es):")
        for v in versions:
            print(f"   - Versión: {v.version}, Aliases: {v.aliases}")
        
        # Verificar si existe el alias 'latest-databricks'
        has_alias = any('latest-databricks' in v.aliases for v in versions)
        
        if not has_alias:
            print("\n⚠️ El alias 'latest-databricks' no existe.")
            print("   Esto es normal si el modelo nunca se registró con ese alias.")
            print("   Puedes asignarlo manualmente con:")
            print(f"   client.set_registered_model_alias('{model_name}', 'latest-databricks', <numero_version>)")
        else:
            print("\n✅ El alias 'latest-databricks' ya existe.")
            
        # Mostrar la versión más reciente
        latest_version = max(int(v.version) for v in versions)
        print(f"\n📌 Última versión registrada: {latest_version}")
        
except Exception as e:
    print(f"❌ Error: {e}")

In [0]:
try:
    logger.log_stage_start('load_model', mes_vta=MES_VTA, environment=ENV, model_path=MODEL_FILE_PATH)
    start_time = time.time()

    logger.info("=" * 60)
    logger.info("CARGANDO MODELO DESDE PICKLE")
    logger.info("=" * 60)

    # ==========================================================
    # 1. Validar existencia del archivo
    # ==========================================================
    if not os.path.exists(MODEL_FILE_PATH):
        raise FileNotFoundError(f"No se encontró el archivo de modelo en: {MODEL_FILE_PATH}")

    file_size = os.path.getsize(MODEL_FILE_PATH)
    if file_size == 0:
        raise ValueError(f"El archivo {MODEL_FILE_PATH} está vacío (0 bytes).")

    # ==========================================================
    # 2. Cargar modelo
    # ==========================================================
    with open(MODEL_FILE_PATH, "rb") as f:
        model = pickle.load(f)

    logger.info(f"✅ Modelo cargado exitosamente desde: {MODEL_FILE_PATH}")
    logger.info(f"Tipo de modelo: {type(model).__name__}")

    # ==========================================================
    # 3. Validar que el modelo tenga predict_proba
    # ==========================================================
    if not hasattr(model, 'predict_proba'):
        raise AttributeError("El modelo cargado no tiene el método 'predict_proba'")

    # ==========================================================
    # 4. Obtener nombres de features (si el modelo los expone)
    # ==========================================================
    feature_names = None
    if hasattr(model, 'feature_names_') and model.feature_names_ is not None:
        feature_names = model.feature_names_
    elif hasattr(model, 'feature_names_in_') and model.feature_names_in_ is not None:
        feature_names = list(model.feature_names_in_)

    if feature_names:
        logger.info(f"Features extraídos del modelo: {len(feature_names)}")
        logger.debug(f"Lista: {feature_names[:5]}..." if len(feature_names) > 5 else f"Lista: {feature_names}")
        dbutils.jobs.taskValues.set(key="features_list", value=str(feature_names))
    else:
        logger.warning("El modelo no expone nombres de features. Se usará FEATURE_NAMES desde config.")
        if 'FEATURE_NAMES' in globals() and FEATURE_NAMES:
            feature_names = FEATURE_NAMES
            logger.info(f"Usando FEATURE_NAMES desde config: {len(feature_names)} variables")
            dbutils.jobs.taskValues.set(key="features_list", value=str(FEATURE_NAMES))
        else:
            logger.warning("No se encontró FEATURE_NAMES en config. La inferencia requerirá la lista manual.")
    
    # ==========================================================
    # 5. Registrar en MLflow (si está configurado)
    # ==========================================================
    registered_version = None
    run_id = None

    if REGISTERED_MODEL_NAME and REGISTERED_MODEL_NAME.strip():
        logger.info(f"Registrando modelo en Unity Catalog: {REGISTERED_MODEL_NAME}")

        # Si no tenemos nombres de features, no podemos crear input_example ni firma
        if not feature_names:
            logger.error("No se pueden obtener nombres de features. No se registrará el modelo.")
        else:
            try:
                # Crear input_example y firma
                input_example = pd.DataFrame([[0.0] * len(feature_names)], columns=feature_names)
                proba_example = model.predict_proba(input_example)[:, 1]
                signature = infer_signature(input_example, proba_example)

                with mlflow.start_run(run_name=f"load_model_{MES_VTA}") as run:
                    run_id = run.info.run_id
                    mlflow.catboost.log_model(
                        cb_model=model,
                        artifact_path="model",
                        registered_model_name=REGISTERED_MODEL_NAME,
                        signature=signature,
                        input_example=input_example
                    )
                    logger.info(f"Modelo registrado | Run ID: {run_id} | Nombre: {REGISTERED_MODEL_NAME}")

                    # Obtener versión registrada (Unity Catalog usa alias)
                    client = MlflowClient()
                    try:
                        latest = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "latest-databricks")
                        registered_version = latest.version
                        logger.info(f"Versión registrada: {registered_version}")
                    except Exception as e_alias:
                        logger.warning(f"No se pudo obtener la versión automáticamente: {e_alias}")
                        logger.info("El modelo se registró correctamente (versión no disponible para lectura)")

            except Exception as e_reg:
                logger.error(f"Error al registrar en MLflow: {e_reg}")
                logger.warning("El modelo no se registró, pero se usará localmente.")
    else:
        logger.info("REGISTERED_MODEL_NAME no definido. No se registrará el modelo.")

    # ==========================================================
    # 6. Task values y finalización
    # ==========================================================
    duration = time.time() - start_time
    logger.log_stage_end('load_model', duration=duration, model_loaded=True)

    dbutils.jobs.taskValues.set(key="model_loaded", value=True)
    dbutils.jobs.taskValues.set(key="model_type", value=type(model).__name__)
    dbutils.jobs.taskValues.set(key="model_version", value=registered_version if registered_version else "none")
    if run_id:
        dbutils.jobs.taskValues.set(key="mlflow_run_id", value=run_id)

    logger.info(f"✅ Modelo cargado exitosamente. Tiempo: {duration:.2f}s")

except Exception as e:
    logger.log_exception(operation='load_model', exception=e, should_raise=True, mes_vta=MES_VTA, environment=ENV)
    dbutils.jobs.taskValues.set(key="model_loaded", value=False)

finally:
    if 'logger' in locals():
        logger.info(f"Finalizando notebook: {logger.name}")
        logger._flush_all_handlers()
        logger.close()